# 03: Preprocess IDRiD (disease grading)

Crops the 516 images of the IDRiD disease grading set (Nanded, India) to the fundus, resizes them
to 512 px and writes them as JPEG into a zip shard, plus a validated manifest. IDRiD grading is
used only as an external test set. Then looks for near-duplicates within IDRiD and between IDRiD
and the sources processed before it (EyePACS, APTOS).

IDRiD: Porwal et al., IEEE DataPort, https://doi.org/10.21227/H25W98, CC BY 4.0.

**Settings** (right-hand panel): Accelerator **None**, Internet **On**.
**Inputs** (*Add Input*):
- *Datasets* → *Your Datasets* → your private upload of `B. Disease Grading`
- *Notebooks* → *Your Work* → **01: Preprocess EyePACS** and **02: Preprocess APTOS**

**Run:** *Save Version* → *Save & Run All (Commit)*. Takes about 10 min.

**Output** (`/kaggle/working/idrid512/`):
- `idrid_000.zip`: processed images, with `.records.parquet`
- `manifest_idrid.parquet`: labels + image columns, schema-validated
- `idrid_duplicate_candidates.parquet`: phash-close pairs involving IDRiD, with similarity
- `failures.csv`, `summary.json`, `spot_check.png`, `idrid_top_pairs.png`

> RetinaGuard is a research prototype, **not a medical device**.

In [ ]:
REPO_URL = "https://github.com/FidaMarzouk/retinaguard.git"
GIT_REF = "main"
OUT_DIR = "/kaggle/working/idrid512"
SEED = 0

In [ ]:
%pip install -q "retinaguard @ git+{REPO_URL}@{GIT_REF}" "pandera>=0.20"

In [ ]:
# --- 1. Locate IDRiD, its labels and the earlier outputs ------------------------
import subprocess
import time
from pathlib import Path

import pandas as pd

from retinaguard import __version__
from retinaguard.data.sources import idrid

INPUT_ROOT = Path("/kaggle/input")
root = idrid.find_root(INPUT_ROOT)
earlier = {}
for source in ("eyepacs", "aptos"):
    found = sorted(INPUT_ROOT.glob(f"**/manifest_{source}.parquet"))
    assert found, f"manifest_{source}.parquet not found: attach the output of its notebook"
    earlier[source] = found[0]

git_sha = subprocess.run(
    ["git", "ls-remote", REPO_URL, GIT_REF], capture_output=True, text=True
).stdout.split("\t")[0]
print(f"retinaguard {__version__} @ {git_sha[:10]}")

images = idrid.GradingImages(root)
labels = idrid.read_labels(root)
missing = set(labels["image_id"]) - images.ids()
print(f"{len(images.ids()):,} images, {len(labels):,} labels")
print("grades:", labels["dr_grade"].value_counts().sort_index().to_dict())
assert not missing, f"{len(missing)} labelled images missing, e.g. {sorted(missing)[:5]}"

In [ ]:
# --- 2. Process all images ------------------------------------------------------
from retinaguard.data.processing import build_shard, shard_ids

out_dir = Path(OUT_DIR)
shards = shard_ids(labels["image_id"])
all_records = []
start = time.perf_counter()
for i, ids in enumerate(shards):
    t = time.perf_counter()
    records = build_shard(out_dir, f"idrid_{i:03d}.zip", ids, images.read)
    all_records.append(records)
    print(
        f"shard {i + 1}/{len(shards)}: {len(ids)} images, "
        f"{records['error'].notna().sum()} failed, {time.perf_counter() - t:5.0f}s",
        flush=True,
    )
records = pd.concat(all_records, ignore_index=True)

In [ ]:
# --- 3. Manifest, failures and summary ---------------------------------------
import json

from retinaguard.data.invariants import duplicate_image_ids
from retinaguard.data.processing import attach_records

manifest = attach_records(labels, records)
assert not duplicate_image_ids(manifest)
manifest.to_parquet(out_dir / "manifest_idrid.parquet", index=False)

failures = records[records["error"].notna()]
failures.to_csv(out_dir / "failures.csv", index=False)

ok = records[records["error"].isna()]
radius_ratio = 2 * ok["fundus_radius"] / ok["width"]
sizes = (ok["width"].astype(str) + "x" + ok["height"].astype(str)).value_counts()
summary = {
    "retinaguard_version": __version__,
    "git_sha": git_sha,
    "preprocess_version": str(records["preprocess_version"].iloc[0]),
    "labelled_images": len(labels),
    "processed": len(ok),
    "failed": len(failures),
    "failure_reasons": failures["error"].str.split(":").str[0].value_counts().to_dict(),
    "exact_duplicate_raw_files": int(records["sha256"].duplicated().sum()),
    "raw_sizes": sizes.head(10).to_dict(),
    "radius_ratio_p01_p50_p99": [
        round(float(q), 3) for q in radius_ratio.quantile([0.01, 0.5, 0.99])
    ],
    "minutes": round((time.perf_counter() - start) / 60, 1),
}
(out_dir / "summary.json").write_text(json.dumps(summary, indent=2))
print(json.dumps(summary, indent=2))
print(failures[["image_id", "error"]].head(20).to_string())

In [ ]:
# --- 4. Spot check: a few processed images straight from the shards ------------
import matplotlib.pyplot as plt

from retinaguard.data.preprocess import local_mean_normalise
from retinaguard.data.processing import ProcessedImages, shard_roots

done = manifest.dropna(subset=["processed_path"])
before = pd.concat([pd.read_parquet(path) for path in earlier.values()], ignore_index=True).dropna(
    subset=["processed_path"]
)
paths = dict(zip(done["image_id"], done["processed_path"], strict=True))
paths.update(zip(before["image_id"], before["processed_path"], strict=True))
folders = [out_dir, *(path.parent for path in earlier.values())]
reader = ProcessedImages(paths, shard_roots(*folders))

sample = done.sample(4, random_state=SEED)
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for col, (_, row) in enumerate(sample.iterrows()):
    img = reader.read(row["image_id"])
    for ax, shown in zip(axes[:, col], (img, local_mean_normalise(img)), strict=True):
        ax.imshow(shown)
        ax.set_title(f"{row['image_id']} grade {row['dr_grade']}", fontsize=8)
        ax.axis("off")
plt.tight_layout()
fig.savefig(out_dir.parent / "spot_check.png", dpi=80)
plt.show()

In [ ]:
# --- 5. Near-duplicates within IDRiD and against the earlier sources ------------
from functools import cache

import numpy as np

from retinaguard.data.duplicates import (
    CANDIDATE_MAX_DISTANCE,
    DUPLICATE_MIN_SIMILARITY,
    candidate_frame,
    score_candidates,
    structure_signature,
    structure_similarity,
)


@cache
def signature(image_id):
    return structure_signature(reader.read(image_id))


def source_of(image_ids):
    return image_ids.str.split(":").str[0]


everything = pd.concat([done, before], ignore_index=True)
t0 = time.perf_counter()
cand = candidate_frame(
    everything["image_id"].tolist(), everything["phash"].tolist(), involving="idrid"
)
pair_sources = pd.concat(
    [source_of(cand["image_id_a"]), source_of(cand["image_id_b"])], axis=1
).to_numpy()
cand["kind"] = ["-".join(sorted(pair)) for pair in pair_sources]
cand = score_candidates(cand, signature)
print(
    f"{len(cand):,} candidates (phash <= {CANDIDATE_MAX_DISTANCE}) scored in "
    f"{time.perf_counter() - t0:.0f}s"
)

# Sanity check of the threshold on this site: random IDRiD-other pairs are different eyes.
rng = np.random.default_rng(SEED)
a = rng.choice(done["image_id"].to_numpy(), 2000)
b = rng.choice(before["image_id"].to_numpy(), 2000)
random_pairs = np.array(
    [structure_similarity(signature(x), signature(y)) for x, y in zip(a, b, strict=True)]
)
print(
    "random IDRiD-other similarity min/p50/p99/max:",
    [round(float(v), 3) for v in np.quantile(random_pairs, [0, 0.5, 0.99, 1])],
)

confirmed = cand[cand["similarity"] >= DUPLICATE_MIN_SIMILARITY]
print(
    f"confirmed duplicates (similarity >= {DUPLICATE_MIN_SIMILARITY}):",
    confirmed["kind"].value_counts().to_dict(),
)

grade = dict(zip(everything["image_id"], everything["dr_grade"], strict=True))
confirmed = confirmed.assign(
    grade_a=confirmed["image_id_a"].map(grade), grade_b=confirmed["image_id_b"].map(grade)
)
for kind, rows in confirmed.groupby("kind"):
    differ = rows["grade_a"] != rows["grade_b"]
    print(f"{kind}: {len(rows)} pairs, {int(differ.sum())} with different grades")
print(confirmed.sort_values("similarity", ascending=False).head(20).to_string())
cand.to_parquet(out_dir / "idrid_duplicate_candidates.parquet", index=False)
summary["duplicates"] = {
    "candidates": cand["kind"].value_counts().to_dict(),
    "confirmed": confirmed["kind"].value_counts().to_dict(),
    "random_idrid_other_max": round(float(random_pairs.max()), 3),
}
_ = (out_dir / "summary.json").write_text(json.dumps(summary, indent=2))

In [ ]:
# --- 6. Look at the most similar pairs of each kind ----------------------------
top = cand.sort_values("similarity", ascending=False).groupby("kind").head(4)
if len(top):
    fig, axes = plt.subplots(len(top), 2, figsize=(6, 3 * len(top)), squeeze=False)
    for row, (_, pair) in zip(axes, top.iterrows(), strict=True):
        for ax, image_id in zip(row, (pair["image_id_a"], pair["image_id_b"]), strict=True):
            ax.imshow(reader.read(image_id))
            ax.set_title(f"{image_id}  grade {grade[image_id]}", fontsize=8)
            ax.axis("off")
        row[0].text(
            0,
            -0.12,
            f"{pair['kind']}: similarity {pair['similarity']:.3f}, "
            f"phash distance {pair['phash_distance']}",
            transform=row[0].transAxes,
            fontsize=8,
        )
    plt.tight_layout()
    fig.savefig(out_dir.parent / "idrid_top_pairs.png", dpi=80)
    plt.show()

**After the version finishes:** download `idrid512/manifest_idrid.parquet` and
`idrid512/idrid_duplicate_candidates.parquet` from the **Output** tab into
`data/interim/idrid/`.